In [13]:
import csv
import os


def is_number(value):
    try:
        float(value)
        return True
    except ValueError:
        return False


def quote(text):
    if text == "" or any(c in text for c in " ,{}%'\"\t"):
        return "'" + text.replace("\\", "\\\\").replace("'", "\\'") + "'"
    return text


def csv_to_arff(csv_path, arff_path=None):
    if arff_path is None:
        arff_path = os.path.splitext(csv_path)[0] + ".arff"

    with open(csv_path, newline="") as f:
        rows = [row for row in csv.reader(f) if row]

    header = [name.strip() for name in rows[0]]
    data = [[value.strip() for value in row] for row in rows[1:]]

    for i, row in enumerate(data, start=2):
        if len(row) != len(header):
            raise ValueError(f"Line {i} has {len(row)} values but the header has {len(header)} names.")

    attributes = []
    for col, name in enumerate(header):
        values = [row[col] for row in data if row[col] not in ("", "?")]
        if values and all(is_number(v) for v in values):
            attributes.append(f"@attribute {quote(name)} numeric")
        else:
            unique = sorted(set(values))
            attributes.append(f"@attribute {quote(name)} {{{','.join(quote(v) for v in unique)}}}")

    relation = os.path.splitext(os.path.basename(csv_path))[0]
    with open(arff_path, "w", newline="\n") as out:
        out.write(f"@relation {quote(relation)}\n\n")
        out.write("\n".join(attributes) + "\n\n@data\n")
        for row in data:
            out.write(",".join("?" if v in ("", "?") else quote(v) for v in row) + "\n")

    print(f"Wrote {arff_path} ({len(data)} rows, {len(header)} attributes)")

In [14]:
csv_to_arff(r"C:/platform-tools-latest-windows/platform-tools/FANZCJ011945436-handwash_new_gui-Deeksha-null-null-null-1970-02-13-06-16-02.csv", r"C:/platform-tools-latest-windows/platform-tools/wada-work/Features.arff")

Wrote C:/platform-tools-latest-windows/platform-tools/wada-work/Features.arff (329 rows, 6 attributes)


The first line of every raw CSV is a single info value, not column names. The function below removes that line and adds a header row (timestamp, sensor_type, accuracy, x, y, z) so the file can be converted. The original CSV is left unchanged; the cleaned copy is saved with `_clean` added to its name.

In [15]:
path = r"C:/platform-tools-latest-windows/platform-tools/FANZCJ011945436-handwash_new_gui-Deeksha-null-null-null-1970-02-13-06-16-02.csv"
with open(path) as f:
    for _ in range(5):
        print(f.readline().rstrip())

3782799881,1,3,-4.1948767,6.192323,5.874983
3782799901,1,3,-4.1565566,6.175558,5.8654027
3782799921,1,3,-4.133804,6.255791,5.9276733
3782800039,1,3,-4.191284,6.0953245,5.7588243
3782800058,1,3,-4.160149,6.1180773,5.853428


In [16]:
COLUMN_NAMES = ["timestamp", "sensor_type", "accuracy", "x", "y", "z"]


def clean_wada_csv(csv_path, clean_path=None):
    """Remove the info line at the top of a raw WADA CSV and add column names."""
    if clean_path is None:
        clean_path = os.path.splitext(csv_path)[0] + "_clean.csv"

    with open(csv_path, newline="") as f:
        lines = f.read().splitlines()

    with open(clean_path, "w", newline="") as out:
        out.write(",".join(COLUMN_NAMES) + "\n")
        for line in lines[1:]:
            if line.strip():
                out.write(line + "\n")

    print(f"Removed info line from {os.path.basename(csv_path)} -> {clean_path}")
    return clean_path

Now calling the same `csv_to_arff` function on the cleaned file converts it to ARFF.

In [17]:
raw_csv = r"C:/platform-tools-latest-windows/platform-tools/FANZCJ011945436-handwash_new_gui-Deeksha-null-null-null-1970-02-13-06-16-02.csv"
out_dir = r"C:/platform-tools-latest-windows/platform-tools/wada-work"

os.makedirs(out_dir, exist_ok=True)
clean_csv = clean_wada_csv(raw_csv)
csv_to_arff(clean_csv, os.path.join(out_dir, "Features.arff"))

Removed info line from FANZCJ011945436-handwash_new_gui-Deeksha-null-null-null-1970-02-13-06-16-02.csv -> C:/platform-tools-latest-windows/platform-tools/FANZCJ011945436-handwash_new_gui-Deeksha-null-null-null-1970-02-13-06-16-02_clean.csv
Wrote C:/platform-tools-latest-windows/platform-tools/wada-work\Features.arff (329 rows, 6 attributes)
